# PHASE 2: DATA SPLITTING AND GROUNDTRUTH CREATION

## 1. Objectives
- Split the raw dataset (N = 250) into two independent subsets: a manual annotation set (100 samples) and an automated processing set (150 samples).
- Conduct independent labeling on the 100-sample annotation set to calculate the inter-annotator agreement score between two team members.
- Resolve annotation conflicts to establish a GroundTruth dataset for model evaluation.

In [1]:
%pip install openpyxl pandas numpy matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import pandas as pd
import numpy as np

# Path to the raw data file
RAW_DATA_PATH = "../data/01_raw/Data_Raw.csv"
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Raw dataset loaded successfully. Current shape: {df_raw.shape}")

Raw dataset loaded successfully. Current shape: (250, 6)


In [17]:
# Randomly sample 100 rows for manual annotation (Ground Truth)
df_100_ground_truth = df_raw.sample(n=100, random_state=42)

# Extract the remaining 150 rows by dropping the sampled indices
df_150_remaining = df_raw.drop(df_100_ground_truth.index)

print(f"Total rows in raw data: {len(df_raw)}")
print(f"Total rows for Ground Truth: {len(df_100_ground_truth)}")
print(f"Total rows remaining for automated pipeline: {len(df_150_remaining)}")

# Export two datasets
df_100_ground_truth.to_excel('../data/02_annotation/100__unannotated_samples.xlsx', index=False)
df_150_remaining.to_csv('../data/03_interim/150_jobs_for_pipeline.csv', index=False)

print("\nData successfully split and exported!")

Total rows in raw data: 250
Total rows for Ground Truth: 100
Total rows remaining for automated pipeline: 150

Data successfully split and exported!


## 2. Independent Annotation and Agreement Evaluation

To build a reliable GroundTruth, both annotators worked through a Modified Blind Annotation process with the following steps:

1. **Independent Annotation:** Two annotators (Dinh Hoang Phong and Nguyen Dinh Tuan Phuc) each worked on `100__unannotated_samples.xlsx` separately, without seeing each other's work. Their results were saved as two files: `100__annotated_samples_Phong.csv` and `100__annotated_samples_Phuc.csv`.
2. **Input Standardization:** Both annotators used dropdown validation in Google Sheets, backed by a predefined skill whitelist (78 classes), to prevent inconsistent text formatting.
3. **Agreement Scoring:**
   - For single-value fields (`Job_Domain`, `Min_years_of_exp`, `Language_Requirement`), the team used Cohen's Kappa to measure how often
     both annotators agreed, adjusted for the probability of random agreement.
   - For the multi-value field (`Skills`), the team calculated Jaccard Similarity for each record and averaged the scores across all 100 records to measure how closely the two skill sets matched.

In [18]:
from sklearn.metrics import cohen_kappa_score, accuracy_score

# Load the two independent annotation files
df_phong = pd.read_csv("../data/02_annotation/100__annotated_samples_Phong.csv")
df_phuc = pd.read_csv("../data/02_annotation/100__annotated_samples_Phuc.csv")
print("Annotation files from both annotators loaded successfully.")

Annotation files from both annotators loaded successfully.


In [19]:
# Preprocessing function: convert skill strings to sets for Jaccard computation
def clean_skills_to_set(skill_str):
    if pd.isna(skill_str) or str(skill_str).strip() == '':
        return set()
    return set([s.strip().lower() for s in str(skill_str).split(',') if s.strip()])

def calculate_jaccard(s1, s2):
    set1 = clean_skills_to_set(s1)
    set2 = clean_skills_to_set(s2)
    if not set1 and not set2: return 1.0
    if not set1 or not set2: return 0.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

# Compute Jaccard score for each record
jaccard_scores = [calculate_jaccard(df_phong['Skills'].iloc[i], df_phuc['Skills'].iloc[i]) for i in range(len(df_phong))]
mean_jaccard_skills = np.mean(jaccard_scores)

print(f"- Cohen's Kappa (Job Domain): {cohen_kappa_score(df_phong['Job Domain'].fillna('None').astype(str), df_phuc['Job Domain'].fillna('None').astype(str)):.4f}")
print(f"- Cohen's Kappa (Min Exp):    {cohen_kappa_score(df_phong['Min years of exp'].fillna('None').astype(str), df_phuc['Min years of exp'].fillna('None').astype(str)):.4f}")
print(f"- Cohen's Kappa (Language):   {cohen_kappa_score(df_phong['Language Requirement'].fillna('None').astype(str), df_phuc['Language Requirement'].fillna('None').astype(str)):.4f}")
print(f"- Mean Jaccard Similarity (Skills): {mean_jaccard_skills * 100:.2f}%")

- Cohen's Kappa (Job Domain): 0.8871
- Cohen's Kappa (Min Exp):    0.8716
- Cohen's Kappa (Language):   0.8923
- Mean Jaccard Similarity (Skills): 73.81%


## 3. GroundTruth Construction

To ensure no information is lost from the source data, two annotators manually reviewed all 100 records together.

**Procedure:**
1. **Cross-review:** Both annotators went through all 100 records side by side, comparing their labels for each field directly.
2. **Discussion and Resolution:** When the two annotators disagreed on a record, especially in multi-value fields such as `Skills`, they returned to the original job description text and consulted the Annotation Guidebook to settle on a final label.
3. **GroundTruth Export:** Once both annotators reached full agreement on all 100 records, the final labels were saved as `GroundTruth_100.csv`. This file is used as the reference output when measuring how well the automated extraction system (LLM) performs in later pipeline stages.

In [20]:
GROUND_TRUTH_PATH = "../data/02_annotation/GroundTruth_100.csv"
df_gt = pd.read_csv(GROUND_TRUTH_PATH)

print("GROUND TRUTH VALIDATION RESULTS:")
print(f"- Shape: {df_gt.shape[0]} rows, {df_gt.shape[1]} columns.")

# Verify that the dataset contains exactly 100 records
assert df_gt.shape[0] == 100, "Error: Ground Truth set does not contain 100 rows!"
print("- Row count check: Passed (100/100 samples).")

print("\nFIRST 10 RECORDS:")
display(df_gt[['Job_Title', 'Job Domain', 'Min years of exp', 'Language Requirement', 'Skills']].head(10))

GROUND TRUTH VALIDATION RESULTS:
- Shape: 100 rows, 11 columns.
- Row count check: Passed (100/100 samples).

FIRST 10 RECORDS:


,Job_Title,Job Domain,Min years of exp,Language Requirement,Skills
0,"Analytics Engineer (Advanced SQL, Python, Data...",Data & AI,3.0,NaN,"SQL, Python, Data Engineering, Data Warehousin..."
1,Data Engineer,Data & AI,3.0,English,"AWS, GCP, SQL, Python, Data Engineering, Data ..."
2,Data Scientist,Data & AI,5.0,NaN,"Python, SQL, Machine Learning, Data Analysis, ..."
3,Data Analyst,Data & AI,0.5,NaN,"Business Analysis, Data Analysis, Data Enginee..."
4,AI/NLP Engineer / Algorithm Engineer,Data & AI,1.0,English,"API, Big Data, GenAI / LLM, Machine Learning, ..."
5,AI Engineer,Data & AI,2.0,NaN,"API, C/C++, Computer Vision, Deep Learning, Dj..."
6,AI Engineer,Data & AI,0.0,NaN,"GenAI / LLM, RAG, Python, SQL, Langchain"
7,Data Engineer,Data & AI,3.0,NaN,"AWS, Business Analysis, Data Analysis, Data En..."
8,AI Engineer (Machine Learning),Data & AI,5.0,NaN,"AWS, Azure, CI/CD, Computer Vision, Deep Learn..."
9,AI Solutions Architect,Data & AI,8.0,NaN,"API, GenAI / LLM, Langchain, JavaScript, Java,..."
